# Build the broad-organic combined melting-point dataset

This notebook combines four prepared input datasets from `input_datasets/` and writes the reproducible outputs to `outputs/`:

- `combined_data.parquet`: resolved, broad-organic, single-component compounds with consensus `MP` in 0–500 °C and RDKit average molecular weight `MW < 1000` Da;
- `multiple_MP_compounds.csv`: an observation-level audit of canonical compounds that have more than one distinct reported MP before the final MP-range and MW filters.

The source labels retain their established order: 1 = ChemXplore (Marimuthu, 2025), 2 = EGNN (Huang, 2025), 3 = PATENTS/OCHEM (Tetko, 2016), and 4 = Bradley data prepared from Austermeier (2025). Generate Source 4 first with `data_preparation.ipynb`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors

RDLogger.DisableLog('rdApp.*')

data_dir_candidates = [Path.cwd(), Path.cwd() / '0_data', Path.cwd().parent / '0_data']
DATA_DIR = next((path.resolve() for path in data_dir_candidates
                 if (path / 'input_datasets').is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not locate 0_data/input_datasets. Run this notebook from the repository root '
        'or from the 0_data directory.'
    )

INPUT_DIR = DATA_DIR / 'input_datasets'
OUTPUT_DIR = DATA_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA = OUTPUT_DIR / 'combined_data.parquet'
OUTPUT_CONFLICTS = OUTPUT_DIR / 'multiple_MP_compounds.csv'
MAX_MW = 1000.0

INPUT_SPECS = [
    (1, '1_ChemXplore_tmpC_Marimuthu_2025.csv', 'tmp/ºC', 'chemxplore'),
    (2, '2_EGNN_mp_Huang_2025.csv', 'Melt_C', 'numeric'),
    (3, '3_PATENTS_ochem_Tetko_2016.parquet',
     'Melting Point {measured, converted}', 'numeric_or_range'),
    (4, '4_Bradley_Austermeier_2025.csv', 'MP', 'numeric'),
]

missing_files = [filename for _, filename, _, _ in INPUT_SPECS
                 if not (INPUT_DIR / filename).is_file()]
if missing_files:
    source4_hint = (
        ' Run data_preparation.ipynb first if 4_Bradley_Austermeier_2025.csv is missing.'
        if '4_Bradley_Austermeier_2025.csv' in missing_files else ''
    )
    raise FileNotFoundError(f'Missing required input file(s): {missing_files}.{source4_hint}')

schema_rows = []
for source_id, filename, mp_column, _ in INPUT_SPECS:
    path = INPUT_DIR / filename
    columns = (pq.read_schema(path).names if path.suffix.lower() == '.parquet'
               else pd.read_csv(path, nrows=0).columns.tolist())
    missing_columns = sorted({'SMILES', mp_column}.difference(columns))
    if missing_columns:
        raise ValueError(f'{path.name} is missing required column(s): {missing_columns}')
    schema_rows.append({
        'Source': source_id, 'Filename': filename,
        'SMILES_column': 'SMILES', 'MP_column': mp_column,
    })

print(f'Input directory: {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
display(pd.DataFrame(schema_rows))


## Curation rules

- Canonical isomeric SMILES are generated with RDKit and must parse successfully after canonicalization.
- ChemXplore values are parsed from the original `tmp/ºC` field. Numeric values with parenthetical uncertainty are retained; censored or approximate values (`<`, `>`, `≈`) are excluded.
- Strict numeric PATENTS ranges are converted to their midpoint. The later consensus statistics describe accepted processed observations across sources, not the endpoints of an individual PATENTS range.
- Broad-organic filtering removes metal-containing and multi-component structures but retains halogen-containing compounds.
- RDKit average molecular weight is calculated from the canonical structure. The final dataset applies the strict condition `MW < 1000` Da.
- MP quality is `single`, `exact`, `near_exact` (spread ≤1 °C), `high_confidence` (>1–5 °C), or `review` (>5–10 °C).
- Groups spanning more than 10 °C are accepted only when at least two sources form a dominant cluster spanning no more than 5 °C and supporting at least two-thirds of the observations. Otherwise they remain unresolved.
- The median of accepted observations is the final `MP`. Calculations use unrounded values.


In [ ]:
NUMBER = r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)'
STRICT_RANGE = re.compile(rf'^\s*({NUMBER})\s*[-–]\s*({NUMBER})\s*$')
CHEMXPLORE_NUMBER = re.compile(rf'^\s*({NUMBER})(?:\(\d+\))?\s*$')

HALOGEN_ATOMIC_NUMBERS = {9, 17, 35, 53, 85, 117}
METAL_SYMBOLS = {
    'Li', 'Be', 'Na', 'Mg', 'Al', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe',
    'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru',
    'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm',
    'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W',
    'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'Fr', 'Ra', 'Ac',
    'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No',
    'Lr', 'Rf', 'Db', 'Sg', 'Bh', 'Hs', 'Mt', 'Ds', 'Rg', 'Cn', 'Nh', 'Fl', 'Mc', 'Lv'
}
periodic_table = Chem.GetPeriodicTable()
METAL_ATOMIC_NUMBERS = {periodic_table.GetAtomicNumber(symbol) for symbol in METAL_SYMBOLS}

def parse_mp(value, method):
    if pd.isna(value):
        return np.nan, 'missing'
    text = str(value).strip()
    if method == 'chemxplore':
        match = CHEMXPLORE_NUMBER.fullmatch(text)
        return (float(match.group(1)), 'numeric') if match else (np.nan, 'censored_or_approximate')
    try:
        return float(text), 'numeric'
    except ValueError:
        if method == 'numeric_or_range':
            match = STRICT_RANGE.fullmatch(text)
            if match:
                return (float(match.group(1)) + float(match.group(2))) / 2.0, 'range_midpoint'
        return np.nan, 'nonnumeric'

def canonicalize_and_screen(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None, False, False, 0, np.nan
        canonical = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        roundtrip = Chem.MolFromSmiles(canonical)
        if roundtrip is None:
            return None, False, False, 0, np.nan
        canonical = Chem.MolToSmiles(roundtrip, canonical=True, isomericSmiles=True)
        atomic_numbers = {atom.GetAtomicNum() for atom in roundtrip.GetAtoms()}
        return (
            canonical,
            bool(atomic_numbers & HALOGEN_ATOMIC_NUMBERS),
            bool(atomic_numbers & METAL_ATOMIC_NUMBERS),
            len(Chem.GetMolFrags(roundtrip)),
            float(Descriptors.MolWt(roundtrip)),
        )
    except Exception:
        return None, False, False, 0, np.nan

def quality_from_values(values, n_sources):
    spread = float(np.max(values) - np.min(values))
    if len(values) == 1 and n_sources == 1:
        return 'single'
    if spread == 0:
        return 'exact'
    if spread <= 1:
        return 'near_exact'
    if spread <= 5:
        return 'high_confidence'
    if spread <= 10:
        return 'review'
    return 'unresolved'

In [ ]:
frames = []
load_audit = []
for source_id, filename, mp_column, method in INPUT_SPECS:
    path = INPUT_DIR / filename
    if path.suffix.lower() == '.parquet':
        raw = pd.read_parquet(path, columns=['SMILES', mp_column])
    else:
        raw = pd.read_csv(path, usecols=['SMILES', mp_column], dtype={mp_column: 'string'})
    parsed = raw[mp_column].map(lambda value: parse_mp(value, method))
    frame = pd.DataFrame({
        'Original_SMILES': raw['SMILES'].astype('string'),
        'MP_raw': raw[mp_column].astype('string'),
        'MP_reported': parsed.map(lambda result: result[0]),
        'Parse_status': parsed.map(lambda result: result[1]),
        'Source': np.int8(source_id),
        'Source_file': filename,
    })
    usable = frame['Original_SMILES'].notna() & frame['MP_reported'].notna()
    load_audit.append({
        'Source_file': filename, 'Source': source_id, 'Input_rows': len(frame),
        'Usable_MP_rows': int(usable.sum()), 'Excluded_MP_rows': int((~usable).sum()),
        'Range_midpoints': int(frame['Parse_status'].eq('range_midpoint').sum()),
    })
    frames.append(frame.loc[usable])
observations = pd.concat(frames, ignore_index=True)
load_audit = pd.DataFrame(load_audit)
display(load_audit)
print(f'Usable observations before structure processing: {len(observations):,}')

In [ ]:
unique_raw_smiles = observations['Original_SMILES'].drop_duplicates().tolist()
structure_results = [canonicalize_and_screen(smiles) for smiles in unique_raw_smiles]
structure_lookup = pd.DataFrame(
    structure_results,
    columns=['SMILES', 'ContainsHalogen', 'ContainsMetal', 'NumFragments', 'MW'],
)
structure_lookup.insert(0, 'Original_SMILES', unique_raw_smiles)
observations = observations.merge(structure_lookup, on='Original_SMILES', how='left', validate='many_to_one')
invalid_structure = observations['SMILES'].isna()
metal_structure = observations['ContainsMetal'].fillna(False)
multicomponent_structure = observations['NumFragments'].fillna(0).ne(1)
broad_organic = ~invalid_structure & ~metal_structure & ~multicomponent_structure
structure_audit = pd.DataFrame({
    'Reason': ['Invalid/non-round-trippable SMILES', 'Contains metal', 'Multiple components',
               'Broad-organic observations retained'],
    'Count': [int(invalid_structure.sum()), int(metal_structure.sum()),
              int(multicomponent_structure.sum()), int(broad_organic.sum())],
})
display(structure_audit)
observations = observations.loc[broad_organic].copy()
observations['Source'] = observations['Source'].astype('int8')
rows_before_exact_deduplication = len(observations)
observations = observations.drop_duplicates(
    ['SMILES', 'MP_reported', 'Source', 'Source_file']
).reset_index(drop=True)
print(f'Exact observation duplicates removed: '
      f'{rows_before_exact_deduplication - len(observations):,}')
print(f'Unique broad-organic canonical SMILES: {observations["SMILES"].nunique():,}')

In [ ]:
def select_dominant_cluster(group, max_span=5.0, minimum_fraction=2/3):
    ordered = group.sort_values('MP_reported', kind='stable')
    best_index, best_score = None, None
    for start in range(len(ordered)):
        for stop in range(start + 1, len(ordered) + 1):
            candidate = ordered.iloc[start:stop]
            spread = candidate['MP_reported'].max() - candidate['MP_reported'].min()
            if spread > max_span:
                break
            score = (candidate['Source'].nunique(), len(candidate), -spread)
            if best_score is None or score > best_score:
                best_score, best_index = score, candidate.index
    if best_index is None:
        return None
    candidate = group.loc[best_index]
    supported = (len(candidate) >= 2 and candidate['Source'].nunique() >= 2
                 and len(candidate) / len(group) >= minimum_fraction)
    return candidate.index if supported else None

consensus_rows, conflict_audit_rows = [], []
for smiles, group in observations.groupby('SMILES', sort=False):
    group = group.copy()
    all_values = group['MP_reported'].to_numpy(dtype=float)
    all_spread = float(all_values.max() - all_values.min())
    has_multiple_mp = group['MP_reported'].nunique() > 1
    if all_spread <= 10:
        accepted_index = group.index
        method = 'single_measurement' if len(group) == 1 else 'median_all'
    else:
        accepted_index = select_dominant_cluster(group)
        method = 'dominant_cluster' if accepted_index is not None else 'unresolved'
    if accepted_index is not None:
        accepted = group.loc[accepted_index]
        values = accepted['MP_reported'].to_numpy(dtype=float)
        sources = sorted({int(source) for source in accepted['Source']})
        quality = quality_from_values(values, len(sources))
        median = float(np.median(values))
        consensus_rows.append({
            'SMILES': smiles, 'MP': median, 'Source': sources,
            'MP_mean': float(np.mean(values)),
            'MP_std': float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            'MP_mad': float(np.median(np.abs(values - median))),
            'MP_min': float(np.min(values)), 'MP_max': float(np.max(values)),
            'MP_range': float(np.max(values) - np.min(values)),
            'N_MP_values': len(values), 'N_distinct_MP': int(accepted['MP_reported'].nunique()),
            'N_sources': len(sources), 'MP_quality': quality, 'Consensus_method': method,
            'Has_excluded_values': len(accepted) < len(group),
            'N_excluded_values': len(group) - len(accepted), 'All_MP_range': all_spread,
            'ContainsHalogen': bool(group['ContainsHalogen'].iloc[0]),
            'MW': float(group['MW'].iloc[0]),
        })
    else:
        quality = 'unresolved'
    if has_multiple_mp:
        accepted_set = set(accepted_index) if accepted_index is not None else set()
        for idx, observation in group.iterrows():
            conflict_audit_rows.append({
                'SMILES': smiles, 'MP_reported': float(observation['MP_reported']),
                'MP_raw': observation['MP_raw'], 'Source': int(observation['Source']),
                'Source_file': observation['Source_file'],
                'Accepted_for_consensus': idx in accepted_set, 'MP_quality': quality,
                'Consensus_method': method, 'Group_MP_range': all_spread,
            })
consensus = pd.DataFrame(consensus_rows)
conflict_audit = pd.DataFrame(conflict_audit_rows)
unresolved_count = conflict_audit.loc[
    conflict_audit['MP_quality'].eq('unresolved'), 'SMILES'
].nunique()
print(f'Resolved canonical compounds before final MP/MW filters: {len(consensus):,}')
print(f'Unresolved canonical compounds: {unresolved_count:,}')

In [ ]:
in_mp_range = consensus['MP'].between(0, 500, inclusive='both')
below_mw_limit = consensus['MW'].lt(MAX_MW)
removed_by_mp = int((~in_mp_range).sum())
removed_by_mw_after_mp = int((in_mp_range & ~below_mw_limit).sum())

combined_data = (
    consensus.loc[in_mp_range & below_mw_limit]
    .sort_values(['SMILES', 'MP'], kind='stable').reset_index(drop=True)
)
conflict_audit = conflict_audit.sort_values(
    ['SMILES', 'MP_reported', 'Source'], kind='stable'
).reset_index(drop=True)

table = pa.Table.from_pandas(combined_data, preserve_index=False)
source_index = table.column_names.index('Source')
table = table.set_column(
    source_index, 'Source',
    pa.array(combined_data['Source'].tolist(), type=pa.list_(pa.int8()))
)
pq.write_table(table, OUTPUT_DATA, compression='zstd')
conflict_audit.to_csv(OUTPUT_CONFLICTS, index=False)

quality_summary = (
    combined_data['MP_quality'].value_counts()
    .reindex(['single', 'exact', 'near_exact', 'high_confidence', 'review'], fill_value=0)
    .rename_axis('MP_quality').reset_index(name='Compounds')
)
display(quality_summary)
print(f'Resolved compounds before final filters: {len(consensus):,}')
print(f'Removed outside 0–500 °C: {removed_by_mp:,}')
print(f'Removed with MW >= {MAX_MW:,.0f} Da after MP filtering: {removed_by_mw_after_mp:,}')
print(f'Final broad-organic compounds: {len(combined_data):,}')
print(f'Halogen-containing compounds retained: {int(combined_data["ContainsHalogen"].sum()):,}')
print(f'Final MW range: {combined_data["MW"].min():.3f}–{combined_data["MW"].max():.3f} Da')
print(f'Wrote: {OUTPUT_DATA}')
print(f'Wrote: {OUTPUT_CONFLICTS}')


In [ ]:
saved_table = pq.read_table(OUTPUT_DATA)
saved = saved_table.to_pandas()
saved_conflicts = pd.read_csv(OUTPUT_CONFLICTS)

expected_columns = [
    'SMILES', 'MP', 'Source', 'MP_mean', 'MP_std', 'MP_mad', 'MP_min', 'MP_max',
    'MP_range', 'N_MP_values', 'N_distinct_MP', 'N_sources', 'MP_quality',
    'Consensus_method', 'Has_excluded_values', 'N_excluded_values',
    'All_MP_range', 'ContainsHalogen', 'MW',
]
assert saved.columns.tolist() == expected_columns
assert saved['SMILES'].is_unique
assert saved['MP'].between(0, 500, inclusive='both').all()
assert saved['MW'].lt(MAX_MW).all()
assert set(saved['MP_quality']).issubset(
    {'single', 'exact', 'near_exact', 'high_confidence', 'review'}
)
assert pa.types.is_list(saved_table.schema.field('Source').type)
assert pa.types.is_int8(saved_table.schema.field('Source').type.value_type)
assert saved['Source'].map(lambda values: list(values) == sorted(set(values))).all()
assert saved['Source'].map(lambda values: set(values).issubset({1, 2, 3, 4})).all()
assert (saved['N_sources'] == saved['Source'].map(len)).all()
assert pd.api.types.is_integer_dtype(saved_conflicts['Source'])

for smiles, contains_halogen, stored_mw in zip(
    saved['SMILES'], saved['ContainsHalogen'], saved['MW']
):
    mol = Chem.MolFromSmiles(smiles)
    assert mol is not None
    assert Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True) == smiles
    assert len(Chem.GetMolFrags(mol)) == 1
    atomic_numbers = {atom.GetAtomicNum() for atom in mol.GetAtoms()}
    assert not (atomic_numbers & METAL_ATOMIC_NUMBERS)
    assert bool(atomic_numbers & HALOGEN_ATOMIC_NUMBERS) == bool(contains_halogen)
    assert np.isclose(Descriptors.MolWt(mol), stored_mw, rtol=0, atol=1e-9)

display(saved.head())
print('Saved schema:')
print(saved_table.schema)
print('All validation checks passed.')
